# D-FINE-L на 5 семейных фолдах (bot-5)
**Только честный замер. Ничего не идёт в сдачу до письма организаторов.**
Код: `scripts/det_train_hf.py` (свой, на HF transformers). Подробности — `kaggle/README_KAGGLE.md`.

Inputs (Kaggle Datasets): `expds-koz3-bot5` (репо, ветка `bot5-dfine-kaggle`), `koz3-spor-det05` (папки `f0..f4`, `grid_f0..grid_f4` из `coco_from_cache.py`). Accelerator: GPU T4×2, Internet ON (веса HF).

In [ ]:
!pip install -q -U "transformers>=4.57" timm 2>&1 | tail -1
import torch, transformers; print(torch.__version__, transformers.__version__, torch.cuda.device_count())

In [ ]:
import os
REPO='/kaggle/input/expds-koz3-bot5'   # поправить на фактический путь датасета
DET='/kaggle/input/koz3-spor-det05'
OUT='/kaggle/working/dfine'
os.makedirs(OUT, exist_ok=True)
!ls $DET

## Шаг 1 — smoke (10 итераций, обязателен первым). Записать `s_per_iter` и `cuda_peak_MB` из timing.json.

In [ ]:
!python $REPO/scripts/det_train_hf.py --data $DET/f0 --out $OUT/smoke --model dfine-l --size 640 --fp16 --smoke
!cat $OUT/smoke/timing.json

Бюджет: `s_per_iter × ITERS × 3` (полоса GPU0 — 3 фолда) + сетка. Если > ~11 ч — ITERS=6000 или 3 фолда за сессию (запустить lane по частям).

In [ ]:
ITERS=8000
!bash $REPO/kaggle/dfine_folds.sh $DET $OUT dfine-l $ITERS

In [ ]:
!for k in 0 1 2 3 4; do cat $OUT/f$k/timing.json | python -c "import json,sys;d=json.load(sys.stdin);print($k,{x:d.get(x) for x in ['s_per_iter','train_s','cuda_peak_MB','loss_last']})"; done
!cd /kaggle/working && tar czf dfine_preds.tgz dfine/f*/grid/preds_valid.json dfine/f*/timing.json dfine/f*/grid/timing.json dfine/*.log && ls -la dfine_preds.tgz

Скачать `dfine_preds.tgz` (Output). Дальше — на машине с данными: `README_KAGGLE.md`, шаг 4 (coco_preds_to_stack → scoreboard --honest 10 → П1–П5).